<a href="https://colab.research.google.com/github/davidmkidd/UK-Supermarket-Carbon-Emissions/blob/main/UKSmktComp_Stores_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://evoviz.uk/wp-content/uploads/2026/04/Food_Divider_trans2.png">

# Clean Geolytix Retailer Points

Emissions and energy use are expected increases with business 'size'. In this second Workbook, the [Geolytix Retail Points] (https://geolytix.com/blog/supermarket-retail-points/) dataset is cleaned and summarised ready for graphical explortation. Data is imported and manipulated using a combination of basic R functions and functions from external libraries, *dpylr* and *tidyr*. Code syntax is explained when first used.

# Set-up

Workbooks begin with a 'set-up' section that loads required R libraries and data.


## Load Libraries
Libraries provide additional 'specialist' functionality not supported by the 'base' program. Colab provides some commonly used libraries as 'ready-to-import' packages loaded with the *library()* function.

* [dpylr](https://dplyr.tidyverse.org/) is data manipulation library.
* [tidyr](https://tidyr.tidyverse.org/) manages ta

In [ ]:
library(dplyr)
library(tidyr)

## Download and Import Data
The next step is to import the Geolytix Retail Points dataset from the Github repository where it is stored to the /content folder.



In [ ]:
# Store the address of the csv file in a variable named 'url'
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/geolytix_retailpoints_v34_202412.csv"

# Store the path in Colab where the file will be saved to
download_path <- "/content/geolytix_retailpoints_v34_202412.csv"

# Download the file to the destination
download.file(url, destfile = download_path, mode = "wb")

#Import the CSV file to a variable named 'store'
store <- read.csv("/content/geolytix_retailpoints_v34_202412.csv", header=TRUE, stringsAsFactors=FALSE)

That is all the set-up required for this workbook.

# Data Frame

*read.csv()* imports data in CSV formatted text file into an R *data frame*. CSV and the R data frame are both 'table' formats in which data is stored as attribute columns and record rows. Let's explore the 'store' data frame.

In [ ]:
# ncol() returns the number of data frame columns
ncol(store)

In [ ]:
# names() and colnames() returns a vector of column names
names(store)

Items in a vector are accessed via their position index []

In [ ]:
# First item
names(store)[1]

# Items 4 to 8
names(store)[4:8]

# Without the third item
names(store)[-3]

In [ ]:
# nrow() returns the number of data frame records
# Each row is a store
nrow(store)

In [ ]:
# summary() returns a summary report of columns and their contents
summary(store)

In [ ]:
# Typing the name of a data frame returns a default number of rows.
store

In [ ]:
# head() returns a set number of the first records.
head(store, n = 5)

In [ ]:
# tail() returns a set number of the last records.
tail(store, n = 5)

In [ ]:
# Values in columns are accessed with the '$' character.

store$retailer

These functions are all 'base' R commands. Libraries provide different functioality that build on each other to perform higher-level operations with minmial consistent syntax.

# Introducing *dpylr*

*dpylr* defines a powerful syntax that provides a simpler more consistent way of working with tabular data that always returns a table, specifically a type of table called a '[tibble](https://tibble.tidyverse.org/reference/tibble.html)', which is an 'extended' data table. The details do not matter.

For example,

In [ ]:
# dpylr (count) returns the number of rows
count(store)

This command may, however also be written as,
```
store %>%
  count()
```
'%>%' passes the result of one statement to another allowing them to be srung together to undertake complex queries and manipulations. Here the content of 'store' is passed to count.

In [ ]:
store %>%
  count()

Emissions data is available for eleven retailers, so one of the first questions to ask is which retailers are in the Retail Points and how many stores each has?

In [ ]:
# dpylr distinct() returns unique values.

store %>% distinct(retailer)

In [ ]:
# To count the number of unique retailers chain store, distinct(), and count()
store %>%
  distinct(retailer) %>%
  count()

Counting the number of stores each retailer has and their percentage is an example of a more complex dpylr query that was 'vibe' coded by asking Gemeni to,

> *"Count the number of store rows for each retailer and calculate the percent of records for each"*

Helpfully, Gemeni added a sort statement despite not being requested.

In [ ]:
# To count the number of stores each retailer has and calculate their percentage
store %>%
  group_by(retailer) %>%
  summarise(count = n(), .groups = "drop") %>%
  mutate(percent = count / sum(count) * 100) %>%
  arrange(desc(percent))

# Clean Store Area

In the Geolytix dataset, stores are assigned to one of five size classes encoded as both *size_level* and *size_band* fields.

In [ ]:
# Unique size levels
store %>%
  distinct(size_level)

# Unique size bands
store %>%
  distinct(size_band)


The base table() fuction is a useful way of getting the number of records for each combination of class pairs.

In [ ]:
# base table() function counts records by one or more variable
table(store$size_band, store$size_level, useNA = "always")

The number of  size band and level records correspond, but one record is has NA values. Let's find out what it is.

In [ ]:
# is.na() finds NAs.
# select(), which limits which columns are returned
store %>%
filter(is.na(size_level)) %>%
    select(retailer, store_name, add_one, suburb, town, postcode, year, size_level, size_band)

The floor area of this store was measured from OS Mastermap data supplied through [Edina Digimap](https://digimap.edina.ac.uk/) at 820m<sup>2</sup>, within *size_level* = 2.

Use dpylr *mutate()* to update 'size_level' and 'size_band'.

In [ ]:
# WARNING
# This code updates the values of ALL records with NA values.
# This may not be approriate if there were more than one NA record.
store <- store %>%
  mutate(size_level = ifelse(is.na(size_level), 2, size_level)) %>%
  mutate(size_band = ifelse(is.na(size_band), "3,013 < 15,069 ft2 (280 < 1,400 m2)", size_band))

# Clean Year
The 'year' field stores the year the store opened.

Use *table()* to count the number of stores recorded as opening each year.

In [ ]:
table(store$year, useNA = "ifany")

There are four things to note:

1. One store is recorded as having opened in 1AD!
2. Far fewer stores are recorded as having opened between 1855 and 2014 than subsequent years.
3. Most stores do not have year recorded.
4. Store closure is not recorded.


A store opening in 1AD is unrealistic, so may be assumed an error and year changed to NA.

In [ ]:
# Update year = 1 to NA
store <- mutate(store, year = ifelse(year == 1, NA, year))

#Check
table(store$year, useNA = "ifany")

The Retail points data set was first released in 2014. Given the difference in  stores recorded as opening, it is reasonable to assume that the opening year of all stores existing in 2014 may not have been known. If this is the case, then 2014 is the **baseline** year to which new stores are added in subsequent years.

In [ ]:
# Copy year to year_built

store$year_built <- store$year

# Update the year values equal or less than 2014 OR (pipe character '|') NA to 2014.
store <- store %>%
  mutate(year = ifelse(year <= 2014 | is.na(store$year), 2014, year))

# Check the output
table(store$retailer, store$year, useNA = "ifany")

* Records with year == 2014 are stores that existed in 2014.
* Records with year >= 2015 are new stores.

# Calculate Summary Statistics
The number and area of retailer stores in each year is calculated by cumulatively summing store number and area by year, with store area approximated as the mid-value of bound store size classes and as the lower boundary of the two upper unbounded classes.

In [ ]:
# Summarise by retailer, year and size_level
store.yr <- store %>%
  count(retailer, year, size_level, sort=TRUE)

In [ ]:
# Download geolytix_retailpoints_v34_202412.csv from GitHub
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/geolytix_size_class.csv"
download_path <- "/content/geolytix_size_class.csv"
download.file(url, destfile = download_path, mode = "wb")

# geolytix_size_class.csv contains the area values that will be used
# for each area class when estimating retailer store area
store.size <- read.csv("/content/geolytix_size_class.csv", header=TRUE, stringsAsFactors=FALSE)
store.size

'Joining' adds the attributes from one table onto another by matching values in fields. Here, the attriributes of store.size are added to store by matching 'size_level' fields.

The dpylr join function is called *merge()*.

In [ ]:
# Join to size class attributes to store.yr
store.yr <- merge(store.yr, store.size, by = "size_level")
head(store.yr, n=2)

Floor area of stores of each size in 2014 and new stores in subsequent years is the product of the number of stores and the size class value.

In [ ]:
# Calculate estimated store area by retailer, year, and level
store.yr$area_m2 <- store.yr$n * store.yr$size_area_mid
head(store.yr, n=2)

Cumulatively sum store number and estimated area by retailer and year.

In [ ]:
# Cumulative Sum of store number by year and retailer
store.yr.n <- store.yr  %>%
  group_by(retailer, year) %>%
  arrange(retailer, year) %>%
  summarise(sum_n = sum(n), .groups = "drop_last") %>%
  mutate(cum_sum_n = cumsum(sum_n))

In [ ]:
# Cumulative Sum of store area by year and retailer
store.yr.a <- store.yr  %>%
  group_by(retailer, year) %>%
  arrange(retailer, year) %>%
  summarise(sum_area_m2 = sum(area_m2), .groups = "drop_last") %>%
  mutate(cum_sum_area_m2 = cumsum(sum_area_m2))

In [ ]:
filter(store.yr.a, retailer == "Spar")

The cumulative sums are not calculated for years in which a retailer did not open any new stores, for example Asda in 2019.

# Reformat and Export

A 'base table' of retailer and year (2014-2024) is created to which values are added.

In [ ]:
# Create 'retailer.yr' of retailer and year 2014-2024
yr <- rep(seq(2014, 2024))
retailers <- distinct(store.yr, retailer)
retailer.yr <- expand.grid(yr, retailers$retailer)
colnames(retailer.yr)[1] <- "year"
colnames(retailer.yr)[2] <- "retailer"
retailer.yr<-data.frame(retailer.yr)

Join cumulative sum tables to retailer.yr on retailer and year fields.

In [ ]:
# Join cumulative sum tables

retailer.yr <- merge(retailer.yr, store.yr.n, by.x = c("retailer","year"), by.y = c("retailer","year"), all = TRUE)
retailer.yr <- merge(retailer.yr, store.yr.a, by.x = c("retailer","year"), by.y = c("retailer","year"), all = TRUE)


head(retailer.yr, n=7)

Use tidyr *fill()* to replace NAs with the value for the previous year.

In [ ]:
# Fill missing Cum Sum values
retailer.yr <- retailer.yr%>%
  arrange(retailer, year) %>%
  fill(cum_sum_n, .direction = "down")

head(retailer.yr, n=10)


In [ ]:
retailer.yr<- retailer.yr %>%
  arrange(retailer, year) %>%
  fill(cum_sum_area_m2, .direction = "down")

In [ ]:
head(retailer.yr, n=2)

In [ ]:
# Rename columns
retailer.yr <- retailer.yr %>% rename("new_store" = "sum_n",
                    "total_store" = "cum_sum_n",
                    "new_area" = "sum_area_m2",
                    "total_area" = "cum_sum_area_m2")

In [ ]:
names(retailer.yr)

The final steps deal with ...

In [ ]:
# Download retailer data
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/retailer_data.csv"
download_path <- "/content/retailer_data.csv"
download.file(url, destfile = download_path, mode = "wb")
retailer.data <- read.csv("/content/retailer_data.csv", header=TRUE, stringsAsFactors=FALSE)

In [ ]:
# Add Retailer Code
retailer.yr <- merge(retailer.yr, retailer.data, by = "retailer", all.x = TRUE)

In [ ]:
# Set New Store and New Area which are NA to zero except for Ocardo

retailer.yr <- retailer.yr %>%
  mutate(new_store = ifelse(retailer != "Ocado" & is.na(new_store), 0, new_store)) %>%
  mutate(new_area = ifelse(retailer != "Ocado" & is.na(new_area), 0, new_area))

In [ ]:
# Set Ocado Total Store and Area to NA

retailer.yr <- retailer.yr %>%
  mutate(total_store = ifelse(retailer == "Ocado", NA, total_store)) %>%
  mutate(total_area = ifelse(retailer == "Ocado", NA, total_area))

In [ ]:
# Check
head(retailer.yr, n = 8)

In [ ]:
retailer.yr %>%
  summarise(total_store = sum(new_store, na.rm = TRUE))

Export cleaned store and store.yr to csv file for next tutorial.

In [ ]:
 write.csv(store,file='/content/geolytix_retailpoints_v34_202412_clean.csv', row.names=FALSE)
 write.csv(retailer.yr,file='/content/geolytix_retailpoints_v34_202412_summary.csv', row.names=FALSE)

---

Back: [UKSmktComp Main Page](https://colab.research.google.com/drive/1f8a0pXfF9PqCujiwjf4TO4-k7ezt-6b3?usp=sharing)

Next: [Visualising Retailer Stores](https://colab.research.google.com/drive/170bNbYyXBmAjgpsRyzqzruE32Jw_5NHR#scrollTo=oUcubW-WW7ok)

---